<a href="https://colab.research.google.com/github/rmkenv/LinkedinWorks/blob/main/llm_zoning_pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ LLM Zoning Pro
**GeoAI Project @rmkenv**

Feed parcel geometries + OSM land use context into an LLM. Flag parcels where the current use seems inconsistent with surrounding zoning — no labeled training data, no ML model, just geospatial feature engineering + Ollama Cloud inference.

**Stack:** `osmnx`, `geopandas`, `shapely`, `folium`, `requests` (Ollama Cloud)  
**Data:** OpenStreetMap (free, no key)  
**LLM:** Ollama Cloud — `gpt-oss:20b` at `https://ollama.com/api`  
**Cost:** Ollama Cloud API usage for ~40 parcels


In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q osmnx geopandas shapely folium matplotlib requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 1.9 MB/s eta 0:00:00


In [2]:
# ── Cell 2: Imports & Ollama client ───────────────────────────────────────────
import os, json, math, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import folium
import requests
import matplotlib.pyplot as plt
from shapely.geometry import mapping
from IPython.display import display
from google.colab import userdata

# ── Ollama Cloud config ────────────────────────────────────────────────────────
# Add OLLAMA_API_KEY to Colab Secrets (key icon in left sidebar)
# Get your key at: https://ollama.com/settings/keys
try:
    OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY')
except:
    OLLAMA_API_KEY = os.environ.get('OLLAMA_API_KEY', 'YOUR_KEY_HERE')

OLLAMA_BASE  = 'https://ollama.com/api'
OLLAMA_MODEL = 'gpt-oss:20b'

def ollama_chat(messages, system=None, expect_json=True):
    """Call Ollama Cloud /api/chat. Returns response content string."""
    all_messages = []
    if system:
        all_messages.append({'role': 'system', 'content': system})
    all_messages.extend(messages)

    payload = {
        'model': OLLAMA_MODEL,
        'messages': all_messages,
        'stream': False
    }
    if expect_json:
        payload['format'] = 'json'

    headers = {
        'Authorization': f'Bearer {OLLAMA_API_KEY}',
        'Content-Type': 'application/json'
    }
    r = requests.post(f'{OLLAMA_BASE}/chat', headers=headers,
                      json=payload, timeout=60)
    r.raise_for_status()
    return r.json()['message']['content']

# Quick connectivity test
try:
    test = ollama_chat([{'role': 'user', 'content': 'Say {"ok": true}'}])
    print(f'✅ Ollama Cloud connected | model: {OLLAMA_MODEL}')
    print(f'   Response: {test[:60]}')
except Exception as e:
    print(f'⚠️  Connection issue: {e}')
    print('   Check your OLLAMA_API_KEY in Colab Secrets')

✅ Ollama Cloud connected | model: gpt-oss:20b
   Response: {"ok": true}


In [14]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
TARGET_PLACE = 'Keene, New Hampshire, USA'  # change to any neighborhood
MAX_BUILDINGS = 40  # keep low to control API calls

ZONE_CONTEXT = """This area is classified as a mixed residential-commercial zone.
Expected land uses: single-family homes, rowhouses, small retail with residential above,
neighborhood services (cafes, barbershops, small grocers). Large industrial footprints,
warehouses, surface parking lots, and out-of-scale commercial buildings are non-conforming."""

print(f'🗺️  Target: {TARGET_PLACE}')
print(f'🏢 Max buildings: {MAX_BUILDINGS}')

🗺️  Target: Keene, New Hampshire, USA
🏢 Max buildings: 40


In [15]:
# ── Cell 4: Pull building footprints from OSM ─────────────────────────────────
print(f'Fetching buildings for {TARGET_PLACE}...')

buildings = ox.features_from_place(TARGET_PLACE, tags={'building': True})
buildings = buildings[buildings.geometry.geom_type.isin(['Polygon','MultiPolygon'])].copy()
buildings = buildings.to_crs('EPSG:3857')
print(f'  Found {len(buildings)} building footprints')

if len(buildings) > MAX_BUILDINGS:
    buildings = buildings.sample(MAX_BUILDINGS, random_state=42)
    print(f'  Sampled {MAX_BUILDINGS} for analysis')

Fetching buildings for Keene, New Hampshire, USA...
  Found 2985 building footprints
  Sampled 40 for analysis


In [16]:
# ── Cell 5: Engineer spatial features ─────────────────────────────────────────
def compactness(geom):
    if geom.is_empty: return None
    return round((4 * math.pi * geom.area) / (geom.length ** 2), 3)

def neighbor_uses(geom, gdf, radius=50):
    buf = geom.buffer(radius)
    nbrs = gdf[gdf.geometry.intersects(buf)]
    uses = []
    for col in ['building','amenity','shop','landuse','office']:
        if col in nbrs.columns:
            vals = nbrs[col].dropna().astype(str).tolist()
            uses += [v for v in vals if v not in ['True','yes','nan']]
    return list(set(uses))[:8]

print('Engineering features...')
records = []
for idx, row in buildings.iterrows():
    geom = row.geometry
    bb = geom.bounds
    tags = {}
    for col in ['building','amenity','shop','landuse','office','name','levels','height']:
        if col in row.index:
            v = row[col]
            if pd.notna(v) and str(v) not in ['True','yes','nan']:
                tags[col] = str(v)
    records.append({
        'geometry': geom,
        'area_m2': round(geom.area, 1),
        'compactness': compactness(geom),
        'elongation': round((bb[2]-bb[0]) / max(bb[3]-bb[1], 1), 2),
        'neighbor_uses': neighbor_uses(geom, buildings),
        'osm_tags': tags,
    })

gdf = gpd.GeoDataFrame(records, geometry='geometry', crs='EPSG:3857').to_crs('EPSG:4326')
print(f'✅ Features ready for {len(gdf)} buildings')

Engineering features...
✅ Features ready for 40 buildings


In [17]:
# ── Cell 6: System prompt ─────────────────────────────────────────────────────
SYSTEM = """You are an expert urban planner and zoning professional reviewing land use
compliance in Mid-Atlantic US cities. Analyze parcel geometry data and surrounding
land use context to identify zoning inconsistencies.

Respond ONLY with valid JSON — no preamble, no markdown fences.
Schema:
{
  "flag": true|false,
  "risk_level": "low"|"medium"|"high",
  "issue_type": "string or null",
  "reasoning": "2-3 sentence explanation",
  "recommended_action": "string or null"
}"""

def build_prompt(row):
    return f"""Zone context:\n{ZONE_CONTEXT}\n\nParcel data:
- Area: {row['area_m2']} m²
- Compactness (0-1): {row['compactness']}
- Elongation: {row['elongation']}
- OSM tags: {json.dumps(row['osm_tags'])}
- Nearby building types (50m): {row['neighbor_uses']}

Flag any zoning inconsistency. Respond with JSON only."""

print('✅ Prompt ready')

✅ Prompt ready


In [18]:
# ── Cell 7: Run Ollama batch inference ────────────────────────────────────────
print(f'Running Ollama Cloud zoning review on {len(gdf)} parcels...\n')

results = []
for i, (_, row) in enumerate(gdf.iterrows()):
    try:
        raw = ollama_chat(
            messages=[{'role': 'user', 'content': build_prompt(row)}],
            system=SYSTEM,
            expect_json=True
        )
        parsed = json.loads(raw.strip())
        parsed['parcel_id'] = i
        results.append(parsed)
        flag_str = '🚩' if parsed.get('flag') else '✅'
        print(f'  [{i+1:02d}/{len(gdf)}] {flag_str} {parsed.get("risk_level","low").upper():6s} | {parsed.get("issue_type") or "none"}')
    except Exception as e:
        print(f'  [{i+1:02d}] ⚠️  {e}')
        results.append({'parcel_id': i, 'flag': False, 'risk_level': 'low',
                        'issue_type': None, 'reasoning': 'Error', 'recommended_action': None})
    time.sleep(0.2)

res_df = pd.DataFrame(results).set_index('parcel_id')
gdf = gdf.reset_index(drop=True)
gdf = pd.concat([gdf, res_df], axis=1)
flagged = gdf[gdf['flag'] == True]
print(f'\n✅ Done — {len(flagged)}/{len(gdf)} parcels flagged')

Running Ollama Cloud zoning review on 40 parcels...

  [01/40] ✅ LOW    | none
  [02/40] ✅ LOW    | none
  [03/40] ✅ LOW    | none
  [04/40] ✅ LOW    | none
  [05/40] ✅ LOW    | none
  [06/40] ✅ LOW    | none
  [07/40] ✅ LOW    | none
  [08/40] ✅ LOW    | none
  [09/40] ✅ LOW    | none
  [10/40] ✅ LOW    | none
  [11/40] ✅ LOW    | none
  [12/40] 🚩 HIGH   | non-conforming industrial use
  [13/40] ✅ LOW    | none
  [14/40] 🚩 HIGH   | Non-conforming use – dormitory
  [15/40] ✅ LOW    | none
  [16/40] ✅ LOW    | none
  [17/40] ✅ LOW    | none
  [18/40] ✅ LOW    | none
  [19/40] ✅ LOW    | none
  [20/40] ✅ LOW    | none
  [21/40] ✅ LOW    | none
  [22/40] 🚩 HIGH   | Non-conforming dormitory
  [23/40] ✅ LOW    | none
  [24/40] ✅ LOW    | none
  [25/40] ✅ LOW    | none
  [26/40] ✅ LOW    | none
  [27/40] ✅ LOW    | none
  [28/40] ✅ LOW    | none
  [29/40] ✅ LOW    | none
  [30/40] ✅ LOW    | none
  [31/40] ✅ LOW    | none
  [32/40] ✅ LOW    | none
  [33/40] ✅ LOW    | none
  [34/40] ✅ LOW   

In [19]:
# ── Cell 8: Summary ───────────────────────────────────────────────────────────
print(f'Total reviewed : {len(gdf)}')
print(f'Flagged        : {len(flagged)}')
print(f'High risk      : {len(gdf[gdf.risk_level=="high"])}')
print(f'Medium risk    : {len(gdf[gdf.risk_level=="medium"])}\n')

for _, row in flagged.sort_values('risk_level').head(5).iterrows():
    print(f'[{row.risk_level.upper()}] {row.get("issue_type","?")} | {row.area_m2:.0f}m²')
    print(f'  {row.reasoning}\n')

Total reviewed : 40
Flagged        : 3
High risk      : 3
Medium risk    : 0

[HIGH] non-conforming industrial use | 7101m²
  The parcel hosts an office tagged as an energy supplier, a use typical of large industrial or commercial operations, which conflicts with the mixed residential‑commercial zoning that limits uses to small retail and residential services. The substantial parcel area and elongated shape further support the interpretation of a non‑conforming industrial facility.

[HIGH] Non-conforming use – dormitory | 3664m²
  The parcel contains a dormitory, which is not an allowed use in the mixed residential‑commercial zone and likely exceeds the scale of permitted structures. The OSM tag confirms a large residential‑type building in close proximity, indicating a potential zoning violation.

[HIGH] Non-conforming dormitory | 793m²
  The parcel hosts a building identified as a dormitory, which is not an allowed use in the mixed residential‑commercial zone where only single‑family

In [20]:
# ── Cell 9: Folium map ────────────────────────────────────────────────────────
center = [gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=15, tiles='CartoDB positron')

risk_colors = {'high': '#d62728', 'medium': '#ff7f0e', 'low': '#2ca02c'}

for _, row in gdf.iterrows():
    color   = risk_colors.get(row.get('risk_level','low'), '#aaa')
    opacity = 0.85 if row.get('flag') else 0.2
    weight  = 3    if row.get('flag') else 1

    popup_html = f"""
    <div style='font-family:monospace;font-size:11px;width:270px'>
    <b>{'🚩 FLAGGED' if row.get('flag') else '✅ OK'}</b> — {str(row.get('risk_level','')).upper()}<br><br>
    <b>Issue:</b> {row.get('issue_type') or 'None'}<br>
    <b>Area:</b> {row['area_m2']:.0f} m² | <b>Compact:</b> {row['compactness']}<br>
    <b>Tags:</b> {row['osm_tags']}<br><br>
    <i>{row.get('reasoning','')}</i><br><br>
    <b>Action:</b> {row.get('recommended_action') or 'None'}
    </div>"""

    folium.GeoJson(
        mapping(row.geometry),
        style_function=lambda x, c=color, o=opacity, w=weight: {
            'fillColor': c, 'color': c, 'weight': w,
            'fillOpacity': o, 'opacity': 1.0
        },
        popup=folium.Popup(popup_html, max_width=290)
    ).add_to(m)

legend = """
<div style='position:fixed;bottom:30px;left:30px;z-index:1000;
background:white;padding:12px;border-radius:8px;
border:1px solid #ccc;font-family:monospace;font-size:11px'>
<b>🏛️ LLM Zoning Lawyer</b><br>
<span style='color:#d62728'>■</span> High risk &nbsp;
<span style='color:#ff7f0e'>■</span> Medium &nbsp;
<span style='color:#2ca02c'>■</span> OK<br>
<i>Powered by Ollama Cloud</i>
</div>"""
m.get_root().html.add_child(folium.Element(legend))
m.save('/content/zoning_map.html')
print('✅ Map saved → /content/zoning_map.html')
display(m)

✅ Map saved → /content/zoning_map.html


In [21]:
# ── Cell 10: Export + LinkedIn hook ──────────────────────────────────────────
out = gdf[['geometry','area_m2','compactness','osm_tags',
           'flag','risk_level','issue_type','reasoning']].copy()
out['osm_tags'] = out['osm_tags'].apply(json.dumps)
out.to_file('/content/zoning_results.geojson', driver='GeoJSON')
gdf[gdf['flag']==True].to_csv('/content/flagged_parcels.csv', index=False)

print('Exported: zoning_results.geojson | flagged_parcels.csv | zoning_map.html')
print()
print('📱 LinkedIn hook:')
print(f'"I built a zoning lawyer out of a map and an LLM.')
print(f' It flagged {len(flagged)} parcels in {TARGET_PLACE.split(",")[0]}')
print(f' in under 2 minutes — no training data, no labels.')
print(f' Here\'s what it found and why planners should care."')

Exported: zoning_results.geojson | flagged_parcels.csv | zoning_map.html

📱 LinkedIn hook:
"I built a zoning lawyer out of a map and an LLM.
 It flagged 3 parcels in Keene
 in under 2 minutes — no training data, no labels.
 Here's what it found and why planners should care."
